# BM25 Ranking System for Legal Document Retrieval
This notebook demonstrates:
1. Verb extraction from questions
2. Concept and relation matching
3. Section retrieval via triplets
4. BM25 ranking of sections

In [2]:

import sys
sys.path.append(r"/")

from src.db import init_mongo
from src.triplet_extraction.pos_taging import init_vncorenlp
import phonlp

E:\Github\LawAssistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Initialize Database and NLP Models

In [3]:
# Initialize MongoDB
mongo_client = init_mongo()
db = mongo_client["KB_PROPERTY_LAW"]
sections_col = db["legal_sections"]
concepts_col = db["concepts"]
relations_col = db["relations"]
triplets_col = db["triplets_new"]

You successfully connected to MongoDB!


In [3]:
# Initialize NLP models
vncorenlp_path = r"/triplet_extraction/nlp_models/VnCoreNLP-1.2"
phonlp_path = r"/triplet_extraction/nlp_models/phonlp"

vncorenlp_client = init_vncorenlp(vncorenlp_path)
phoNLP_model = phonlp.load(save_dir=phonlp_path)

print("NLP models loaded successfully!")

Loading model from: E:\Github\LawAssistant\triplet_extraction\nlp_models\phonlp/phonlp.pt
NLP models loaded successfully!


## 2. Test with a Question

In [4]:
# Test question
question = "Điều kiện chuyển nhượng quyền sử dụng đất là gì?"
print(f"Question: {question}")

Question: Điều kiện chuyển nhượng quyền sử dụng đất là gì?


## 3. Retrieve and Rank Sections

In [5]:
from src.retrieval import retrieve_and_rank

# Retrieve and rank using hybrid approach (BM25 + Triplet scoring)
ranked_results = retrieve_and_rank(
    question=question,
    vncorenlp_client=vncorenlp_client,
    phoNLP_model=phoNLP_model,
    sections_col=db.sections,
    concepts_col=db.concepts,
    relations_col=db.relations,
    triplets_col=db.triplets,
    k_hops=2,
    use_khop=True
)

LEGAL DOCUMENT RETRIEVAL WITH GRAPH TRAVERSAL

Segmented question: điều_kiện chuyển_nhượng quyền sử_dụng đất là gì


100%|██████████| 1/1 [00:00<00:00, 12.18it/s]

Extracted verbs: ['chuyển_nhượng', 'sử_dụng', 'là']

Matched 359 relations
  'chuyển_nhượng' → nhận chuyển nhượng
  'chuyển_nhượng' → chuyển nhượng
  'chuyển_nhượng' → bán
  'chuyển_nhượng' → nhận chuyển nhượng có
  'chuyển_nhượng' → nhận chuyển nhượng nhận
  'chuyển_nhượng' → giải quyết
  'chuyển_nhượng' → là chuyển nhượng
  'chuyển_nhượng' → tổ chức chuyển nhượng
  'chuyển_nhượng' → cho phép chuyển nhượng
  'chuyển_nhượng' → nhận chuyển nhượng để
  'chuyển_nhượng' → chuyển nhượng hợp pháp
  'chuyển_nhượng' → chuyển nhượng thuê
  'chuyển_nhượng' → thoả thuận chuyển nhượng
  'sử_dụng' → sử dụng
  'sử_dụng' → để
  'sử_dụng' → sử dụng do
  'sử_dụng' → sử dụng có
  'sử_dụng' → sử dụng liên kết
  'sử_dụng' → sử dụng để
  'sử_dụng' → có sử dụng
  'sử_dụng' → quy hoạch sử dụng
  'sử_dụng' → sử dụng đồng bộ
  'sử_dụng' → sử dụng quy hoạch
  'sử_dụng' → sử dụng của
  'sử_dụng' → sử dụng quy định
  'sử_dụng' → sử dụng trưng bày
  'sử_dụng' → sử dụng sử dụng
  'sử_dụng' → giải quyết
  'sử_dụng' 

Matched 7579 concepts
  'chuyển_nhượng điều_kiện' → quyền chuyển nhượng điều kiện chào bán
  'chuyển_nhượng điều_kiện' → quyền chuyển nhượng điều kiện là
  'chuyển_nhượng điều_kiện' → quyền chuyển nhượng điều kiện
  'sử_dụng quyền' → quyền sử dụng quyền sở hữu tài sản gắn liền với đất
  'sử_dụng quyền' → quyền sử dụng quyền sử dụng đất
  ... and 7574 more

=== K-HOP TRAVERSAL (k=2) ===
Seed concepts: 7579
Seed relations: 359

Hop 1: Starting with 7574 concepts
Found 9948 triplets at hop 1
Discovered 4268 unique concepts (new: 1739)
Total concepts so far: 9313


=== TRAVERSAL COMPLETE ===
Total concepts: 9313
Total relations: 351
Total triplets: 9948
Sample triplet keys: ['_id', 'subject_id', 'relation_id', 'object_id', 'subject_name', 'relation_name', 'object_name', 'section_id', 'so_hieu']

=== SECTION SCORING ===
Sections found: 2278
Score range: 10.0 - 575.0

Found 2278 candidate sections
Fetched 0 sections for ranking


## 4. Display Results

In [6]:
# Display ranked results
display_results(ranked_sections, sections_col)


=== RANKED RESULTS ===

--- Rank 1 ---
Section ID: c00bb12001d8fb0c3d7d792a4d75e5072809447f708de21cfdc49807071ea359
Hybrid Score: 0.6935
  - BM25 Score: 11.9140 (normalized: 0.8642)
  - Triplet Score: 28 (normalized: 0.4375)
Full Path: 101/2024/NĐ-CP_chương iii_mục 3_điều 30_khoản 13_điểm b
Content Preview: Hợp đồng chuyển nhượng hoặc hợp đồng chuyển giao khác về quyền sử dụng đất, quyền sở hữu tài sản gắn liền với đất giữa người có quyền chuyển nhượng, bán tài sản thế chấp là quyền sử dụng đất, tài sản ...

--- Rank 2 ---
Section ID: 4b66ffea2a2685fbffa06ec3adbf6a0b6940ffe2cb03a930345ae9c3c3216029
Hybrid Score: 0.6250
  - BM25 Score: 13.7862 (normalized: 1.0000)
  - Triplet Score: 4 (normalized: 0.0625)
Full Path: 31/2024/QH15_chương iii_mục 4_điều 44_khoản 3_điểm a
Content Preview: Trong trường hợp chuyển nhượng quyền sử dụng đất thì bên chuyển nhượng trong hợp đồng chuyển nhượng quyền sử dụng đất là người nhận thừa kế;

--- Rank 3 ---
Section ID: 92f29b3a0c8788749049928ec7e0393eddb

## 5. Try BM25-only Ranking (Without Triplet Scores)

In [ ]:
# Retrieve and rank using BM25 only
ranked_sections_bm25 = retrieve_and_rank(
    question=question,
    vncorenlp_client=vncorenlp_client,
    phoNLP_model=phoNLP_model,
    sections_col=sections_col,
    concepts_col=concepts_col,
    relations_col=relations_col,
    triplets_col=triplets_col,
    top_k=10,
    use_hybrid=False  # BM25 only
)

display_results(ranked_sections_bm25, sections_col)

## 6. Compare Different Questions

In [ ]:
test_questions = [
    "Điều kiện để được cấp giấy chứng nhận quyền sử dụng đất là gì?",
    "Ai có quyền chuyển nhượng quyền sử dụng đất?",
    "Thủ tục thu hồi đất để thực hiện dự án đầu tư như thế nào?",
]

for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*100}")
    print(f"QUESTION {i}: {q}")
    print('='*100)
    
    results = retrieve_and_rank(
        question=q,
        vncorenlp_client=vncorenlp_client,
        phoNLP_model=phoNLP_model,
        sections_col=sections_col,
        concepts_col=concepts_col,
        relations_col=relations_col,
        triplets_col=triplets_col,
        top_k=5,
        use_hybrid=True
    )
    
    display_results(results, sections_col)

## 7. View Detailed Concept and Relation Matching

In [ ]:
# Get detailed matching information
from src.retrieval import print_matched_concepts_relations

ranked_sections, matched_concepts, matched_relations = retrieve_and_rank(
    question=question,
    vncorenlp_client=vncorenlp_client,
    phoNLP_model=phoNLP_model,
    sections_col=sections_col,
    concepts_col=concepts_col,
    relations_col=relations_col,
    triplets_col=triplets_col,
    top_k=5,
    use_hybrid=True,
    return_matches=True  # Return matching details
)

# Print detailed matching information
print_matched_concepts_relations(matched_concepts, matched_relations)

## 8. Inspect Individual Components

In [ ]:
# Inspect verb extraction
from src.retrieval import extract_verbs, match_concepts_and_relations
from src.triplet_extraction.src import clean_text

test_q = "Phải xác nhận tài sản trên đất mới được bán đất có đúng không?"
cleaned = clean_text(test_q)
segmented = vncorenlp_client.word_segment(cleaned)[0]

print("Original:", test_q)
print("Cleaned:", cleaned)
print("Segmented:", segmented)

verbs = extract_verbs(segmented, phoNLP_model)
print("\nExtracted Verbs:")
for v in verbs:
    print(f"  - {v['word']} (head: {v['head']}, deprel: {v['deprel']})")

In [ ]:
# Inspect concept/relation matching
segmented_tokens = segmented.split(" ")
all_concepts = list(concepts_col.find({}))
all_relations = list(relations_col.find({}))

matched_concepts, matched_relations = match_concepts_and_relations(
    segmented_tokens, all_concepts, all_relations
)

print("\nMatched Concepts:")
for m in matched_concepts:
    print(f"  Position {m['position']}: '{m['matched_text']}' -> {m['data']['name']}")

print("\nMatched Relations:")
for m in matched_relations:
    print(f"  Position {m['position']}: '{m['matched_text']}' -> {m['data']['name']}")

In [38]:
from typing import Dict, List


def collect_section_content(
    sections_col,
    section_ids: List[str]
) -> Dict[str, str]:
    """
    For each section_id:
      - walk parent_id upward until type == 'điều'
      - collect all content on the path

    Returns:
      { section_id: merged_content }
    """

    pipeline = [
        {
            "$match": {
                "_id": {"$in": section_ids}
            }
        },
        {
            "$graphLookup": {
                "from": sections_col.name,
                "startWith": "$parent_id",
                "connectFromField": "parent_id",
                "connectToField": "_id",
                "as": "ancestors",
                "depthField": "depth"
            }
        },
        {
            "$addFields": {
                "chain": {
                    "$concatArrays": [["$$ROOT"], "$ancestors"]
                }
            }
        },
        {
            "$project": {
                "_id": 1,
                "chain": 1
            }
        }
    ]

    docs = list(sections_col.aggregate(pipeline))

    result = {}

    for doc in docs:
        chain = doc["chain"]

        # sort bottom → top
        chain.sort(key=lambda x: x.get("depth", -1))

        contents = []
        for s in chain:
            if s.get("content"):
                contents.append(s["content"].strip())
            if s.get("type") == "điều":
                break

        result[str(doc["_id"])] = "\n".join(reversed(contents))

    return result

In [6]:
from src.retrieval.utils.collect_content import collect_sections_content_downward

section_ids = [
    "1f95a0cba1d9c7772b506ccd012a25ed66e786e5daeec2af27b8352797135205"
]

output = collect_sections_content_downward(sections_col, section_ids)

print(output[section_ids[0]])

nhận quyền sử dụng đất
Tổ chức trong nước, cá nhân được nhận chuyển nhượng quyền sử dụng đất theo quy định của pháp luật không phụ thuộc vào nơi cư trú, nơi đóng trụ sở, trừ trường hợp quy định tại khoản 8 Điều 45 và Điều 48 của Luật này.
Người nhận quyền sử dụng đất được quy định như sau:
Đối với khu vực hạn chế tiếp cận đất đai thì việc nhận quyền sử dụng đất quy định tại khoản 1 và khoản 2 Điều này thực hiện theo trình tự, thủ tục do Chính phủ quy định.
Tổ chức trong nước là pháp nhân mới được hình thành thông qua việc chia, tách, sáp nhập, hợp nhất, chuyển đổi mô hình tổ chức theo quyết định của cơ quan, tổ chức có thẩm quyền hoặc văn bản về việc chia, tách, sáp nhập, hợp nhất, chuyển đổi mô hình tổ chức của tổ chức kinh tế phù hợp với pháp luật được nhận quyền sử dụng đất từ các tổ chức là pháp nhân bị chia, tách, sáp nhập, hợp nhất, chuyển đổi.
Người gốc Việt Nam định cư ở nước ngoài được phép nhập cảnh vào Việt Nam được mua, thuê mua nhà ở gắn liền với quyền sử dụng đất ở, nhận 